# Feature Engineering and Modeling Notebook

---

### Goal
Build a rich set of customer-level features from raw order data,
train three classification models, and compare their performance.

### What we load
Cleaned data from Day 1: orders_enriched.csv and customer_churn_labels.csv

---

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score,
    RocCurveDisplay, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

print("Libraries ready.")

Libraries ready.


In [15]:
orders_enriched  = pd.read_csv('../data/processed/orders_enriched.csv',
                                parse_dates=['order_purchase_timestamp'])
customer_stats   = pd.read_csv('../data/processed/customer_churn_labels.csv',
                                parse_dates=['last_purchase', 'first_purchase'])
products         = pd.read_csv('../data/raw/olist_products_dataset.csv')
items            = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
reviews          = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
payments         = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')

print(f"orders_enriched:  {orders_enriched.shape}")
print(f"customer_stats:   {customer_stats.shape}")

orders_enriched:  (96478, 11)
customer_stats:   (93358, 7)


### Feature Engineering

Building RFM features first, then behavioral features.
Goal: one row per customer, all features ready for modeling.

In [16]:
snapshot_date = orders_enriched['order_purchase_timestamp'].max() - pd.Timedelta(days=1)

rfm = orders_enriched.groupby('customer_unique_id').agg(
    recency   = ('order_purchase_timestamp', lambda x: (snapshot_date - x.max()).days),
    frequency = ('order_id', 'count'),
    monetary  = ('order_value', 'sum')
).reset_index()

rfm['recency'] = rfm['recency'].clip(lower=0)

print(f"RFM table shape: {rfm.shape}")
print(rfm.describe().round(2))

RFM table shape: (93358, 4)
        recency  frequency  monetary
count  93358.00   93358.00  93358.00
mean     235.94       1.03    165.20
std      152.59       0.21    226.31
min        0.00       1.00      0.00
25%      112.00       1.00     63.05
50%      217.00       1.00    107.78
75%      344.00       1.00    182.56
max      712.00      15.00  13664.08


In [18]:
rfm['avg_order_value'] = rfm['monetary'] / rfm['frequency']

lifespan = (
    orders_enriched
    .groupby('customer_unique_id')['order_purchase_timestamp']
    .apply(lambda x: (x.max() - x.min()).days)
    .reset_index()
)
lifespan.columns = ['customer_unique_id', 'customer_lifespan_days']

rfm = rfm.merge(lifespan, on='customer_unique_id', how='left')

print(rfm[['recency', 'frequency', 'monetary',
           'avg_order_value', 'customer_lifespan_days']].describe().round(2))

        recency  frequency  monetary  avg_order_value  customer_lifespan_days
count  93358.00   93358.00  93358.00         93358.00                93358.00
mean     235.94       1.03    165.20           160.31                    2.63
std      152.59       0.21    226.31           219.57                   24.96
min        0.00       1.00      0.00             0.00                    0.00
25%      112.00       1.00     63.05            62.37                    0.00
50%      217.00       1.00    107.78           105.63                    0.00
75%      344.00       1.00    182.56           176.65                    0.00
max      712.00      15.00  13664.08         13664.08                  633.00


In [19]:
items_products = items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)

items_customers = items_products.merge(
    orders_enriched[['order_id', 'customer_unique_id']],
    on='order_id',
    how='left'
)

category_features = (
    items_customers
    .groupby('customer_unique_id')
    .agg(
        unique_categories = ('product_category_name', 'nunique'),
        total_items       = ('order_item_id', 'count')
    )
    .reset_index()
)

rfm = rfm.merge(category_features, on='customer_unique_id', how='left')

print(f"RFM shape after category features: {rfm.shape}")
print(rfm[['unique_categories', 'total_items']].describe().round(2))

RFM shape after category features: (93358, 10)
       unique_categories  total_items
count           93358.00     93358.00
mean                1.01         1.18
std                 0.20         0.62
min                 0.00         1.00
25%                 1.00         1.00
50%                 1.00         1.00
75%                 1.00         1.00
max                 5.00        24.00


In [20]:
order_reviews = orders_enriched[['order_id', 'customer_unique_id']].merge(
    reviews[['order_id', 'review_score']].drop_duplicates('order_id'),
    on='order_id',
    how='left'
)

review_features = (
    order_reviews
    .groupby('customer_unique_id')
    .agg(
        avg_review_score = ('review_score', 'mean'),
        review_count     = ('review_score', 'count')
    )
    .reset_index()
)

rfm = rfm.merge(review_features, on='customer_unique_id', how='left')

print(f"RFM shape after review features: {rfm.shape}")
print(rfm[['avg_review_score', 'review_count']].describe().round(2))

RFM shape after review features: (93358, 12)
       avg_review_score  review_count
count          92755.00      93358.00
mean               4.15          1.03
std                1.28          0.22
min                1.00          0.00
25%                4.00          1.00
50%                5.00          1.00
75%                5.00          1.00
max                5.00         15.00


In [21]:
payment_features = (
    orders_enriched[['order_id', 'customer_unique_id']]
    .merge(payments[['order_id', 'payment_type', 'payment_installments']],
           on='order_id', how='left')
    .groupby('customer_unique_id')
    .agg(
        avg_installments  = ('payment_installments', 'mean'),
        used_credit_card  = ('payment_type', lambda x: (x == 'credit_card').any().astype(int))
    )
    .reset_index()
)

rfm = rfm.merge(payment_features, on='customer_unique_id', how='left')

print(f"RFM shape after payment features: {rfm.shape}")
print(rfm[['avg_installments', 'used_credit_card']].describe().round(2))

RFM shape after payment features: (93358, 14)
       avg_installments  used_credit_card
count          93357.00          93358.00
mean               2.90              0.77
std                2.68              0.42
min                0.00              0.00
25%                1.00              1.00
50%                2.00              1.00
75%                4.00              1.00
max               24.00              1.00
